In [1]:
import os
import re
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
openai_api = os.getenv("open_api")

llm = OpenAI(api_key = openai_api )
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma = chromadb.Client()
collection = chroma.create_collection("codebase")

In [3]:
REPO_PATH = "F:\Projects\InboxRag_V2"  

In [4]:
def split_by_functions(code):
    pattern = r"(def .*?:|class .*?:)"
    splits = re.split(pattern, code)

    chunks = []
    for i in range(1, len(splits), 2):
        header = splits[i]
        body = splits[i+1] if i+1 < len(splits) else ""
        chunks.append(header + body)

    return chunks

In [5]:
chunks = []

for root, _, files in os.walk(REPO_PATH):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            with open(path, "r", encoding="utf-8") as f:
                code = f.read()

            parts = split_by_functions(code)

            for p in parts:
                chunks.append({
                    "text": p,
                    "file": path
                })

print("Total chunks:", len(chunks))

Total chunks: 49


In [6]:
texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts).tolist()

for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk["text"]],
        metadatas=[{"file": chunk["file"]}]
    )

print("Stored in vector DB")

Stored in vector DB


In [7]:
def retrieve(query, k=5):
    emb = embed_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=emb,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    docs = results["documents"][0]
    metas = results["metadatas"][0]

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    print("\n[DEBUG] Retrieved files:")
    for m in metas:
        print(m["file"])

    context = ""
    for d, m in zip(docs, metas):
        context += f"\nFILE: {m['file']}\nCODE:\n{d}\n"

    return context

In [8]:
def ask(question):
    context = retrieve(question)

    prompt = f"""
You are a senior software engineer analyzing a codebase.

Answer ONLY using the provided context.
Cite file names.
If unsure, say you did not find any relavant contents to the user query

Context:
{context}

Question: {question}
"""

    resp = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return resp.choices[0].message.content

In [9]:
ask("Where is authentication implemented?")


[DEBUG] Retrieved files:
F:\Projects\InboxRag_V2\data_ingestion\gmail_extract.py
F:\Projects\InboxRag_V2\data_ingestion\test.py
F:\Projects\InboxRag_V2\data_ingestion\logger_config.py
F:\Projects\InboxRag_V2\query_retriever\tools\logger_config.py
F:\Projects\InboxRag_V2\query_retriever\guardrails\pre_validation.py


'Authentication is implemented in the following files:\n\n1. **F:\\Projects\\InboxRag_V2\\data_ingestion\\gmail_extract.py**\n2. **F:\\Projects\\InboxRag_V2\\data_ingestion\\test.py**'

In [10]:
ask("Where are API routes defined?")


[DEBUG] Retrieved files:
F:\Projects\InboxRag_V2\query_retriever\app.py
F:\Projects\InboxRag_V2\query_retriever\guardrails\pre_validation.py
F:\Projects\InboxRag_V2\data_ingestion\gmail_extract.py
F:\Projects\InboxRag_V2\data_ingestion\test.py
F:\Projects\InboxRag_V2\query_retriever\tools\vector_store_v2.py


'API routes are defined in the file F:\\Projects\\InboxRag_V2\\query_retriever\\app.py, specifically with the line `@app.route("/get", methods=["POST"])`.'

In [11]:
ask("How does request flow?")


[DEBUG] Retrieved files:
F:\Projects\InboxRag_V2\data_ingestion\gmail_extract.py
F:\Projects\InboxRag_V2\data_ingestion\test.py
F:\Projects\InboxRag_V2\query_retriever\app.py
F:\Projects\InboxRag_V2\query_retriever\app.py
F:\Projects\InboxRag_V2\query_retriever\tools\response_llm.py


'I did not find any relevant contents to the user query regarding request flow.'

In [12]:
ask("What database is used?")


[DEBUG] Retrieved files:
F:\Projects\InboxRag_V2\query_retriever\main.py
F:\Projects\InboxRag_V2\query_retriever\guardrails\pre_validation.py
F:\Projects\InboxRag_V2\query_retriever\tools\vector_store_v2.py
F:\Projects\InboxRag_V2\query_retriever\tools\vector_store_v2.py
F:\Projects\InboxRag_V2\query_retriever\guardrails\pre_validation.py


'I did not find any relevant contents regarding the database used in the provided codebase context.'

In [13]:
ask("What are key modules?")


[DEBUG] Retrieved files:
F:\Projects\InboxRag_V2\query_retriever\agent\controller.py
F:\Projects\InboxRag_V2\query_retriever\guardrails\pre_validation.py
F:\Projects\InboxRag_V2\query_retriever\guardrails\pre_validation.py
F:\Projects\InboxRag_V2\query_retriever\tools\response_llm.py
F:\Projects\InboxRag_V2\data_ingestion\generate_embeddings.py


'The key modules in the provided codebase are:\n\n1. **query_retriever.agent.controller** - Contains the `run_agent` function which handles the querying process.\n2. **query_retriever.guardrails.pre_validation** - Includes the `QueryInputGuardRail` class and an `__init__` function for query management.\n3. **query_retriever.tools.response_llm** - Contains the `fetch_llm_response` function for handling responses from the language model.\n4. **data_ingestion.generate_embeddings** - Contains the `EmbeddingManager` class for embedding management.'